# Judge calibration: compare judge grades to human grades

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from agent.judge import judge
from agent.llm import LLM
from agent.schema import CRITERIA
from agent.trajectory import Trajectory

GRADES_PATH = ROOT / "labeling" / "human_grades.csv"
grades = pd.read_csv(GRADES_PATH)
print(f"{len(grades)} human-graded cases")
grades

## Split: tune vs held-out
Edit TUNE to pick which cases go to tuning. Change USE to `"held_out"` for final validation.

In [ ]:
TUNE = {"c12"}  # add more case ids here as you grade them
USE = "tune"  # change to "held_out" for final validation

held_out = set(grades.case_id) - TUNE
selected = TUNE if USE == "tune" else held_out
subset = grades[grades.case_id.isin(selected)].copy()
print(f"using {USE}: {sorted(selected)} ({len(subset)} cases)")
subset

## Run judge on selected cases

In [ ]:
PROMPT_VERSION = "judge_v1"  # change when you edit the prompt, e.g. "judge_v2"

llm = LLM()
judgements = {}
for _, row in subset.iterrows():
    t = Trajectory(row.request_id)
    j, _ = judge(llm, t.text(), cache_tag=PROMPT_VERSION)
    judgements[row.case_id] = j
    print(f"{row.case_id}: {', '.join(c + '=' + getattr(j, c).result for c in CRITERIA)}")

print(f"\njudged {len(judgements)} cases")

## Compare judge to human

In [ ]:
rows = []
for _, r in subset.iterrows():
    j = judgements[r.case_id]
    row = {"case_id": r.case_id}
    for c in CRITERIA:
        row[f"{c}_human"] = r[c]
        row[f"{c}_judge"] = getattr(j, c).result
        row[f"{c}_agree"] = r[c] == getattr(j, c).result
    rows.append(row)

compare = pd.DataFrame(rows)

# agreement per criterion
agree_cols = [f"{c}_agree" for c in CRITERIA]
agreement = compare[agree_cols].mean().rename(lambda s: s.replace("_agree", ""))
print("agreement per criterion:")
print(agreement.to_string())
print(f"\noverall: {agreement.mean():.0%}")

## Disagreements
Read these to decide what to fix in the judge prompt.

In [ ]:
for c in CRITERIA:
    disagree = compare[~compare[f"{c}_agree"]]
    if disagree.empty:
        continue
    print(f"\n{'=' * 80}")
    print(f"{c}: {len(disagree)} disagreement(s)")
    print("=" * 80)
    for _, row in disagree.iterrows():
        j = judgements[row.case_id]
        grade = getattr(j, c)
        print(f"  {row.case_id}: human={row[f'{c}_human']}  judge={grade.result}")
        print(f"    judge reasoning: {grade.reasoning}")